# BiLSTM + Self-Attention Training Notebook
**Train on Kaggle GPU (T4 x2 or P100) → download weights for local inference**

## Architecture Upgrade
- **OLD**: LSTM(128, 2 layers) → Dropout → Dense(64) → Dense(3)
- **NEW**: BiLSTM(128, 2 layers) → Self-Attention → Dropout → Dense(64) → Dense(3)

## What to do:
1. Enable GPU: Settings → Accelerator → **GPU T4 x2** or P100
2. Run All (~20-30 minutes)
3. Download from `/kaggle/working/`: `bilstm_weights.pth`, `feature_scaler.pkl`, `model_config.json`
4. Place all 3 files in your local `models/pretrained/` directory
5. Restart the Streamlit app

In [ ]:
# Install dependencies
!pip install yfinance ta joblib -q

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import yfinance as yf
import ta
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
SEQ_LEN        = 60       # 60-day lookback (matches lstm_model.py)
HIDDEN_SIZE    = 128      # BiLSTM hidden units (matches lstm_model.py)
NUM_LAYERS     = 2        # BiLSTM layers (matches lstm_model.py)
DROPOUT        = 0.3      # (matches lstm_model.py)
NUM_CLASSES    = 3        # Sell=0, Hold=1, Buy=2
BATCH_SIZE     = 256
NUM_EPOCHS     = 150
LR             = 0.001
BUY_THRESH     = 0.05     # +5% → Buy
SELL_THRESH    = -0.05    # -5% → Sell
HORIZON        = 30       # predict 30-day forward return

# Training universe: broad mix of NSE large/mid caps for generalisation
TICKERS = [
    'RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS', 'ICICIBANK.NS',
    'HINDUNILVR.NS', 'SBIN.NS', 'BAJFINANCE.NS', 'BHARTIARTL.NS', 'WIPRO.NS',
    'AXISBANK.NS', 'LT.NS', 'MARUTI.NS', 'SUNPHARMA.NS', 'M&M.NS',
    'KOTAKBANK.NS', 'ITC.NS', 'HCLTECH.NS', 'ASIANPAINT.NS', 'TITAN.NS',
    'ULTRACEMCO.NS', 'BAJAJFINSV.NS', 'POWERGRID.NS', 'NTPC.NS', 'ONGC.NS',
    'DRREDDY.NS', 'DIVISLAB.NS', 'CIPLA.NS', 'TECHM.NS', 'NESTLEIND.NS',
    'JSWSTEEL.NS', 'TATASTEEL.NS', 'HINDALCO.NS', 'COALINDIA.NS', 'GRASIM.NS',
    'PIIND.NS', 'PERSISTENT.NS', 'COFORGE.NS', 'MPHASIS.NS', 'KPITTECH.NS',
    'ZYDUSLIFE.NS', 'TORNTPHARM.NS', 'GLAND.NS', 'ALKEM.NS', 'ABBOTINDIA.NS',
    'CHOLAFIN.NS', 'MFSL.NS', 'LICHSGFIN.NS', 'ABCAPITAL.NS', 'PNBHOUSING.NS',
    'FEDERALBNK.NS', 'IDFCFIRSTB.NS', 'RBLBANK.NS', 'BANDHANBNK.NS',
]

print(f'Training on {len(TICKERS)} NSE stocks')

In [ ]:
# ─────────────────────────────────────────────────────────────
# DATA PIPELINE
# ─────────────────────────────────────────────────────────────
def build_target(close, horizon=HORIZON):
    """ATR-normalised 3-class target"""
    future_return = close.shift(-horizon) / close - 1.0
    target = pd.Series(1, index=close.index)
    target[future_return > BUY_THRESH]  = 2
    target[future_return < SELL_THRESH] = 0
    target[future_return.isna()]        = np.nan
    return target


def fetch_and_build(ticker):
    try:
        df = yf.download(ticker, period='10y', interval='1d', auto_adjust=True, progress=False)
        if df is None or df.empty or len(df) < 500:
            return None, None
            
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
            
        c = pd.Series(df['Close'].values.flatten(), index=df.index, name='Close', dtype=float)
        h = pd.Series(df['High'].values.flatten(), index=df.index, name='High', dtype=float)
        lo = pd.Series(df['Low'].values.flatten(), index=df.index, name='Low', dtype=float)
        v = pd.Series(df['Volume'].values.flatten(), index=df.index, name='Volume', dtype=float)
        
        X = pd.DataFrame(index=df.index)
        for w in [1, 5, 10, 20, 60]:
            X[f'ret_{w}d'] = np.log(c / c.shift(w))
        for w in [10, 20, 60]:
            X[f'vol_{w}d'] = c.pct_change().rolling(w).std()
            
        X['SMA_20']  = ta.trend.sma_indicator(c, window=20)
        X['SMA_50']  = ta.trend.sma_indicator(c, window=50)
        X['SMA_200'] = ta.trend.sma_indicator(c, window=200)
        X['EMA_12']  = ta.trend.ema_indicator(c, window=12)
        X['EMA_26']  = ta.trend.ema_indicator(c, window=26)
        X['price_vs_sma50']  = (c - X['SMA_50']) / X['SMA_50'].replace(0, np.nan)
        X['price_vs_sma200'] = (c - X['SMA_200']) / X['SMA_200'].replace(0, np.nan)
        
        X['RSI_14']    = ta.momentum.rsi(c, window=14)
        X['StochRSI']  = ta.momentum.stochrsi(c, window=14)
        X['Williams_R']= ta.momentum.williams_r(h, lo, c)
        
        macd = ta.trend.MACD(c)
        X['MACD']        = macd.macd()
        X['MACD_Signal'] = macd.macd_signal()
        X['MACD_Hist']   = macd.macd_diff()
        
        bb = ta.volatility.BollingerBands(c)
        X['BB_PctB'] = bb.bollinger_pband()
        X['ATR_14']  = ta.volatility.average_true_range(h, lo, c, window=14)
        X['Volume_Ratio'] = v / v.rolling(20).mean()
        X['OBV_roc']      = ta.volume.on_balance_volume(c, v).pct_change(20)
        X['ADX_14'] = ta.trend.adx(h, lo, c, window=14)
        X['CCI_20'] = ta.trend.cci(h, lo, c, window=20)
        X['drawdown_52w'] = (c - c.rolling(252).max()) / c.rolling(252).max().replace(0, np.nan)
        
        target = build_target(c)

        X.replace([np.inf, -np.inf], np.nan, inplace=True)
        X.ffill(inplace=True)
        X.fillna(0, inplace=True)

        return X, target
    except Exception as e:
        print(f'  Failed {ticker}: {e}')
        return None, None


# Fetch data for all tickers
all_X, all_y = [], []
feature_cols = None

for ticker in TICKERS:
    print(f'Fetching {ticker}...')
    X, y = fetch_and_build(ticker)
    if X is None:
        continue
    if feature_cols is None:
        feature_cols = list(X.columns)
    all_X.append(X)
    all_y.append(y)

print(f'\nLoaded {len(all_X)} stocks, features: {len(feature_cols) if feature_cols else 0}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# BUILD WINDOWED SEQUENCES
# ─────────────────────────────────────────────────────────────
def build_sequences(X, y, seq_len=SEQ_LEN):
    """Convert a feature DataFrame + target Series into windowed sequences."""
    sequences, labels = [], []
    X_vals = X.values.astype(np.float32)
    y_vals = y.values

    for i in range(seq_len, len(X_vals)):
        lbl = y_vals[i]
        if np.isnan(lbl):
            continue
        seq = X_vals[i - seq_len:i]  # (seq_len, n_features)
        sequences.append(seq)
        labels.append(int(lbl))

    return np.array(sequences), np.array(labels)


X_seqs_list, y_list = [], []
for X, y in zip(all_X, all_y):
    seqs, lbls = build_sequences(X, y)
    X_seqs_list.append(seqs)
    y_list.append(lbls)

X_all = np.concatenate(X_seqs_list, axis=0)  # (N, seq_len, features)
y_all = np.concatenate(y_list, axis=0)        # (N,)

print(f'Total sequences: {len(X_all):,}')
print(f'Shape: {X_all.shape}')
print(f'Class distribution: Sell={np.sum(y_all==0):,}  Hold={np.sum(y_all==1):,}  Buy={np.sum(y_all==2):,}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# SCALE FEATURES
# ─────────────────────────────────────────────────────────────
# Fit scaler on (N, features) by reshaping
n, seq, feats = X_all.shape
X_flat = X_all.reshape(-1, feats)

scaler = RobustScaler()   # RobustScaler is better for financial data (outlier-resistant)
X_scaled_flat = scaler.fit_transform(X_flat)
X_scaled = X_scaled_flat.reshape(n, seq, feats)
X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)

# Save scaler
joblib.dump(scaler, '/kaggle/working/feature_scaler.pkl')
print('Scaler saved.')

# Train/Val split (temporal — last 15% for val)
split = int(0.85 * len(X_scaled))
X_train, X_val = X_scaled[:split], X_scaled[split:]
y_train, y_val = y_all[:split], y_all[split:]

print(f'Train: {len(X_train):,}  Val: {len(X_val):,}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# MODEL: BiLSTM + Self-Attention
# ─────────────────────────────────────────────────────────────
class SelfAttention(nn.Module):
    """Single-head attention over LSTM sequence — MUST MATCH lstm_model.py"""
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size * 2, 1)

    def forward(self, lstm_out):
        scores  = self.attn(lstm_out)
        weights = torch.softmax(scores, dim=1)
        context = (lstm_out * weights).sum(dim=1)
        return context


class BiLSTMNetwork(nn.Module):
    """MUST MATCH lstm_model.py BiLSTMNetwork exactly."""
    def __init__(self, input_size, hidden_size=HIDDEN_SIZE,
                 num_layers=NUM_LAYERS, dropout=DROPOUT, num_classes=NUM_CLASSES):
        super().__init__()
        self.bilstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.attention = SelfAttention(hidden_size)
        self.dropout   = nn.Dropout(dropout)
        self.fc1       = nn.Linear(hidden_size * 2, 64)
        self.relu      = nn.ReLU()
        self.fc2       = nn.Linear(64, num_classes)

    def forward(self, x):
        lstm_out, _ = self.bilstm(x)
        context     = self.attention(lstm_out)
        out         = self.dropout(context)
        out         = self.relu(self.fc1(out))
        return self.fc2(out)


INPUT_SIZE = feats
model = BiLSTMNetwork(input_size=INPUT_SIZE).to(DEVICE)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# TRAINING
# ─────────────────────────────────────────────────────────────
# Class-balanced loss weights
class_weights = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=y_train)
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)
print(f'Class weights: Sell={class_weights[0]:.2f}  Hold={class_weights[1]:.2f}  Buy={class_weights[2]:.2f}')

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# Cosine annealing LR schedule
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)

# DataLoaders
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)
X_val_t   = torch.FloatTensor(X_val)
y_val_t   = torch.LongTensor(y_val)

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds   = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False, pin_memory=True, num_workers=2)


def evaluate(loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out  = model(xb)
            loss_sum += criterion(out, yb).item() * len(yb)
            preds = out.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total   += len(yb)
    return correct / total, loss_sum / total


# Training loop with early stopping
best_val_acc = 0.0
patience, patience_count = 15, 0

print('\nTraining BiLSTM + Self-Attention...')
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        out  = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    val_acc, val_loss = evaluate(val_loader)

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:03d} | Train Loss: {train_loss/len(train_loader):.4f} '
              f'| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/kaggle/working/bilstm_weights.pth')
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= patience:
            print(f'Early stopping at epoch {epoch} (best val acc: {best_val_acc:.4f})')
            break

print(f'\nBest Validation Accuracy: {best_val_acc:.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# DIRECTION ACCURACY (most important metric)
# ─────────────────────────────────────────────────────────────
# Load best weights
model.load_state_dict(torch.load('/kaggle/working/bilstm_weights.pth', map_location=DEVICE))
model.eval()

# Evaluate buy/sell direction accuracy (ignoring Hold)
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(DEVICE)
        out = model(xb)
        preds = out.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# Overall accuracy
overall_acc = (all_preds == all_labels).mean()

# Direction accuracy (Buy vs Sell only, excluding Hold predictions)
directional_mask = (all_preds != 1) & (all_labels != 1)
if directional_mask.sum() > 0:
    dir_preds  = all_preds[directional_mask]
    dir_labels = all_labels[directional_mask]
    dir_acc = (dir_preds == dir_labels).mean()
else:
    dir_acc = 0.0

print(f'Overall Accuracy:   {overall_acc:.4f} ({overall_acc*100:.1f}%)')
print(f'Directional Accuracy: {dir_acc:.4f} ({dir_acc*100:.1f}%)')

# Per-class accuracy
for cls, name in [(0, 'Sell'), (1, 'Hold'), (2, 'Buy')]:
    mask = all_labels == cls
    if mask.sum() > 0:
        cls_acc = (all_preds[mask] == cls).mean()
        print(f'  {name} Recall: {cls_acc:.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# SAVE MODEL CONFIG
# ─────────────────────────────────────────────────────────────
config = {
    'arch':        'bilstm',
    'n_features':  INPUT_SIZE,
    'hidden_size': HIDDEN_SIZE,
    'num_layers':  NUM_LAYERS,
    'dropout':     DROPOUT,
    'seq_len':     SEQ_LEN,
    'num_classes': NUM_CLASSES,
    'val_accuracy': round(float(best_val_acc), 4),
    'dir_accuracy': round(float(dir_acc), 4),
    'feature_cols': feature_cols,
    'train_tickers': TICKERS,
    'horizon_days': HORIZON,
}

with open('/kaggle/working/model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('\n✅ Saved:')
print('  /kaggle/working/bilstm_weights.pth')
print('  /kaggle/working/feature_scaler.pkl')
print('  /kaggle/working/model_config.json')
print()
print('─── NEXT STEPS ──────────────────────────────────────────')
print('1. Download all 3 files from the Output panel (right side)')
print('2. Place them in: models/pretrained/')
print('3. Restart the Streamlit app')
print('4. BiLSTM+Attention will automatically be detected and used')